# Run this script once to prepare the updated gpkg file (GeoPandasDataframe) from the latest film_locations_dataset in SF

## Load Data

In [42]:
"""
Convert the SFgov 2026 film-locations CSV into the canonical project
GeoPackage.

This is a ONE-SHOT script. Run it once. Inspect the output. From that
point forward the pipeline reads the GeoPackage, never this CSV.

What it does:
    1. Reads the SFgov CSV.
    2. Drops the SFgov bookkeeping columns (data_as_of, data_loaded_at).
    3. Renames columns to project snake_case convention.
    4. Builds Point geometry from Longitude/Latitude (null-safe).
    5. Wraps as a GeoDataFrame in EPSG:4326.
    6. Applies the project's street-suffix normalizer to the Locations column.
    7. Writes a GeoPackage at the target path.
    8. Prints a verification summary.

Edit the IN_PATH and OUT_PATH at the top of the script before running.
"""

from pathlib import Path
import re

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# === LOAD DATA ===
from google.colab import drive
drive.mount('/content/drive')
CSV_PATH = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/data-reconcillation-2026/Film_Locations_in_San_Francisco_20260424.csv'
GPD_PATH = '/content/drive/MyDrive/Colab Notebooks/SF Film Project/data-reconcillation-2026/sf_film_2026_04_24_data.gpkg'

# ============================================================================
# CONFIG — edit these two paths before running
# ============================================================================

IN_PATH = Path(CSV_PATH)
OUT_PATH = Path(GPD_PATH)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Street suffix normalizer function definition


In [43]:

# ============================================================================
# Street suffix normalizer
# Identical copy of the function in pipeline_e2e_testing_post_llm_resiliency.ipynb.
# Applied at conversion time so the saved GeoPackage already has normalized
# Locations strings — no re-normalization needed at load time.
# ============================================================================

NORMALIZATION_RULES = [
    # Street
    (re.compile(r'\bStreets?\b', re.IGNORECASE), 'St'),
    (re.compile(r'(?<=[a-zA-Z]\s)St\.?(?!\w)', re.IGNORECASE), 'St'),
    # Avenue
    (re.compile(r'\bAvenues?\b', re.IGNORECASE), 'Ave'),
    (re.compile(r'\bAve\.?(?!\w)', re.IGNORECASE), 'Ave'),
    # Boulevard
    (re.compile(r'\bBoulevards?\b', re.IGNORECASE), 'Blvd'),
    (re.compile(r'\bBlvd\.?(?!\w)', re.IGNORECASE), 'Blvd'),
    # Place
    (re.compile(r'\bPlaces?\b', re.IGNORECASE), 'Pl'),
    (re.compile(r'\bPl\.?(?!\w)', re.IGNORECASE), 'Pl'),
    # Lane
    (re.compile(r'\bLanes?\b', re.IGNORECASE), 'Ln'),
    (re.compile(r'\bLn\.?(?!\w)', re.IGNORECASE), 'Ln'),
    # Road
    (re.compile(r'\bRoads?\b', re.IGNORECASE), 'Rd'),
    (re.compile(r'\bRd\.?(?!\w)', re.IGNORECASE), 'Rd'),
]


def normalize_street_suffixes(text):
    """Apply street-suffix normalization rules to a single string.

    Safe on None/NaN/empty input — returns input unchanged.
    """
    if not text or pd.isna(text):
        return text
    for pattern, replacement in NORMALIZATION_RULES:
        text = pattern.sub(replacement, text)
    return text



## Column data



In [44]:

# ============================================================================
# Column rename map
# Source CSV has spaces in column names. Project convention is snake_case
# with underscores. Drop the SFgov bookkeeping columns.
# ============================================================================

COLUMN_RENAMES = {
    'Title': 'Title',
    'Release Year': 'Year',
    'Locations': 'Locations',
    'Fun Facts': 'Fun_Facts',
    'Production Company': 'Production_Company',
    'Distributor': 'Distributor',
    'Director': 'Director',
    'Writer': 'Writer',
    'Actor 1': 'Actor_1',
    'Actor 2': 'Actor_2',
    'Actor 3': 'Actor_3',
    'Longitude': 'Longitude',
    'Latitude': 'Latitude',
    'Analysis Neighborhood': 'Neighborhood',
    'Supervisor District': 'Supervisor_District',
}

COLUMNS_TO_DROP = ['Point', 'data_as_of', 'data_loaded_at']
# 'Point' is dropped because we rebuild geometry from Longitude/Latitude
# directly — keeping the WKT string would be redundant with the geometry column.


### note: columns=['Longitude', 'Latitude']) will be dropped in the conversion cell

> After this change, code that wants the longitude or latitude as a number reads it from the geometry: gdf.geometry.x for longitude, gdf.geometry.y for latitude.

## conversion op:
 - Drop bookkeeping columns
 - Rename remaining columns to project convention
  - Convert numeric ID columns from float64 to nullable Int64.


In [45]:
# ============================================================================
# Conversion
# ============================================================================
assert IN_PATH.exists(), f'Input file not found: {IN_PATH}'
print(f'Reading {IN_PATH}')

df = pd.read_csv(IN_PATH)
print(f'  Loaded {len(df)} rows, {len(df.columns)} columns')
print(f'  Source columns: {list(df.columns)}')
print()

# --- Drop bookkeeping columns ---------------------------------------
drop_actual = [c for c in COLUMNS_TO_DROP if c in df.columns]
if drop_actual:
    df = df.drop(columns=drop_actual)
    print(f'Dropped columns: {drop_actual}')

# --- Rename to project convention -----------------------------------
df = df.rename(columns=COLUMN_RENAMES)
print(f'Renamed columns to project convention')
print(f'  Columns now: {list(df.columns)}')
print()


# These are categorically integers (years, district numbers) — they only
# became float64 because pandas upgrades int columns containing NaN. The
# Int64 nullable type holds integers AND nulls correctly.
INT_COLUMNS = ['Year', 'Supervisor_District']
for col in INT_COLUMNS:
    df[col] = df[col].astype('Int64')
print(f'Cast to nullable Int64: {INT_COLUMNS}')
print(f'Columns now after casting : {list(df.columns)}')
print(f'Columns\' dtypes : {list(df.dtypes)}')

print()

Reading /content/drive/MyDrive/Colab Notebooks/SF Film Project/data-reconcillation-2026/Film_Locations_in_San_Francisco_20260424.csv
  Loaded 2214 rows, 18 columns
  Source columns: ['Title', 'Release Year', 'Locations', 'Fun Facts', 'Production Company', 'Distributor', 'Director', 'Writer', 'Actor 1', 'Actor 2', 'Actor 3', 'Point', 'Longitude', 'Latitude', 'Analysis Neighborhood', 'Supervisor District', 'data_as_of', 'data_loaded_at']

Dropped columns: ['Point', 'data_as_of', 'data_loaded_at']
Renamed columns to project convention
  Columns now: ['Title', 'Year', 'Locations', 'Fun_Facts', 'Production_Company', 'Distributor', 'Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3', 'Longitude', 'Latitude', 'Neighborhood', 'Supervisor_District']

Cast to nullable Int64: ['Year', 'Supervisor_District']
Columns now after casting : ['Title', 'Year', 'Locations', 'Fun_Facts', 'Production_Company', 'Distributor', 'Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3', 'Longitude', 'Latitude', '

## identify & dedup exact rows

In [46]:
# Identify and drop exact duplicate rows. Use the full set of content
# columns as the dedup key — duplicates on (Title, Year, Locations) alone
# would incorrectly merge rows that differ in Fun_Facts (e.g. San Andreas
# at AT&T Stadium has two scenes with different fun facts at the same
# location).
DEDUP_COLUMNS = [
    'Title', 'Year', 'Locations', 'Fun_Facts',
    'Production_Company', 'Distributor',
    'Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3',
]
n_before = len(df)
df = df.drop_duplicates(subset=DEDUP_COLUMNS, keep='first').reset_index(drop=True)
n_dropped = n_before - len(df)
print(f'Dropped {n_dropped} exact-duplicate rows ({n_before} -> {len(df)})')

Dropped 6 exact-duplicate rows (2214 -> 2208)


- Street normalization
- Build geometry column from lat and lon
- Drop latitude and Longitude columns  

In [47]:

# --- Apply street-suffix normalization to Locations -----------------
n_before_norm = df['Locations'].notna().sum()
df['Locations'] = df['Locations'].apply(normalize_street_suffixes)
n_after_norm = df['Locations'].notna().sum()
assert n_before_norm == n_after_norm, (
    'Street normalization changed null count — should not happen'
)
print(f'Normalized street suffixes on {n_after_norm} non-null Locations strings')
print()

# --- Build Point geometry from Longitude/Latitude -------------------
# Null-safe: rows with missing lat or lon get None geometry, which is
# valid in a GeoDataFrame.
def make_point(row):
    lon, lat = row['Longitude'], row['Latitude']
    if pd.isna(lon) or pd.isna(lat):
        return None
    return Point(lon, lat)

df['geometry'] = df.apply(make_point, axis=1)
n_with_geom = df['geometry'].notna().sum()
n_null_geom = df['geometry'].isna().sum()
print(f'Built geometry: {n_with_geom} rows with Point, {n_null_geom} rows with null geometry')
print()

# Drop the now-redundant numeric columns; geometry is the source of truth
df = df.drop(columns=['Longitude', 'Latitude'])

# --- Wrap as GeoDataFrame in EPSG:4326 ------------------------------
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

print(f'  Columns now          : {list(gdf.columns)}')
print(f'  Columns datatypes now: {list(gdf.dtypes)}')



Normalized street suffixes on 2154 non-null Locations strings

Built geometry: 2122 rows with Point, 86 rows with null geometry

  Columns now          : ['Title', 'Year', 'Locations', 'Fun_Facts', 'Production_Company', 'Distributor', 'Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3', 'Neighborhood', 'Supervisor_District', 'geometry']
  Columns datatypes now: [dtype('O'), Int64Dtype(), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), dtype('O'), Int64Dtype(), <geopandas.array.GeometryDtype object at 0x7ff32e9dde80>]


## verification

In [48]:
# --- Verification summary -------------------------------------------
print('=' * 70)
print('Verification summary')
print('=' * 70)
print(f'Final shape:        {gdf.shape}')
print(f'CRS:                {gdf.crs}')
print(f'Geometry null:      {gdf.geometry.isna().sum()}')
print(f'Neighborhood null:  {gdf["Neighborhood"].isna().sum()}')
print(f'Unique films:       {gdf.drop_duplicates(subset=["Title", "Year"]).shape[0]}')
print()
print('Column dtypes:')
print(gdf.dtypes.to_string())
print()
print('Top 10 neighborhoods (by row count):')
print(gdf['Neighborhood'].value_counts(dropna=False).head(10).to_string())
print()
print('Sample row (with geometry):')
sample = gdf[gdf.geometry.notna()].iloc[0]
for col in gdf.columns:
    val = sample[col]
    if isinstance(val, str) and len(val) > 70:
        val = val[:67] + '...'
    print(f'  {col}: {val}')
print()



Verification summary
Final shape:        (2208, 14)
CRS:                EPSG:4326
Geometry null:      86
Neighborhood null:  144
Unique films:       352

Column dtypes:
Title                    object
Year                      Int64
Locations                object
Fun_Facts                object
Production_Company       object
Distributor              object
Director                 object
Writer                   object
Actor_1                  object
Actor_2                  object
Actor_3                  object
Neighborhood             object
Supervisor_District       Int64
geometry               geometry

Top 10 neighborhoods (by row count):
Neighborhood
Financial District/South Beach    299
North Beach                       202
Chinatown                         163
Nob Hill                          163
Mission                           153
NaN                               144
Tenderloin                        138
Castro/Upper Market                88
Russian Hill                

## write to file

In [49]:
# --- Write -----------------------------------------------------------
print(f'Writing to {OUT_PATH}')
gdf.to_file(OUT_PATH, driver='GPKG')
print(f'  Wrote {len(gdf)} rows')
print()
print('Done. The pipeline can now read this GeoPackage with:')
print(f"  gpd.read_file('{OUT_PATH}')")

Writing to /content/drive/MyDrive/Colab Notebooks/SF Film Project/data-reconcillation-2026/sf_film_2026_04_24_data.gpkg
  Wrote 2208 rows

Done. The pipeline can now read this GeoPackage with:
  gpd.read_file('/content/drive/MyDrive/Colab Notebooks/SF Film Project/data-reconcillation-2026/sf_film_2026_04_24_data.gpkg')


## manual testing

In [55]:
# 1 read the file
gdf = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/SF Film Project/data-reconcillation-2026/sf_film_2026_04_24_data.gpkg')
print(f"Loaded {len(gdf)} rows, {len(gdf.columns)} columns")
print(f"Columns: {list(gdf.columns)}")
print(f"Unique films: {gdf.drop_duplicates(subset=['Title','Year']).shape[0]}")



Loaded 2208 rows, 14 columns
Columns: ['Title', 'Year', 'Locations', 'Fun_Facts', 'Production_Company', 'Distributor', 'Director', 'Writer', 'Actor_1', 'Actor_2', 'Actor_3', 'Neighborhood', 'Supervisor_District', 'geometry']
Unique films: 352


In [54]:
# gdf[gdf.duplicated(subset=['Title', 'Year', 'Locations'], keep=False)]
gdf[gdf['Director'].str.contains('Hitchcock', na=False)]

,Title,Year,Locations,Fun_Facts,Production_Company,Distributor,Director,Writer,Actor_1,Actor_2,Actor_3,Neighborhood,Supervisor_District,geometry
47,Family Plot,1976,2230 Sacramento St,"Called ""1001 Franklin"" in the film.",Universal Pictures,Universal Pictures,Alfred Hitchcock,Ernest Lehman,Karen Black,Bruce Dern,Barbara Harris,Pacific Heights,2,POINT (-122.43062 37.79054)
198,Vertigo,1958,California Palace of the Legion of Honor (34th...,"Built in 1924, the Legion of Honor is a 3/4 re...",Alfred J. Hitchcock Productions,Paramount Pictures,Alfred Hitchcock,Alec Coppel,James Stewart,Kim Novak,Barbara Bel Geddes,Lincoln Park,1,POINT (-122.50084 37.78447)
457,Vertigo,1958,York Hotel (940 Sutter St),NaN,Alfred J. Hitchcock Productions,Paramount Pictures,Alfred Hitchcock,Alec Coppel,James Stewart,Kim Novak,Barbara Bel Geddes,Nob Hill,3,POINT (-122.41594 37.78858)
483,Vertigo,1958,Brocklebank Apartments (1000 Mason St),NaN,Alfred J. Hitchcock Productions,Paramount Pictures,Alfred Hitchcock,Alec Coppel,James Stewart,Kim Novak,Barbara Bel Geddes,Nob Hill,3,POINT (-122.41066 37.79311)
605,Vertigo,1958,900 Lombard St,Lombard Street is not actually the most crooke...,Alfred J. Hitchcock Productions,Paramount Pictures,Alfred Hitchcock,Alec Coppel,James Stewart,Kim Novak,Barbara Bel Geddes,Russian Hill,3,POINT (-122.41657 37.80255)
679,Vertigo,1958,Mission San Juan Bautista (2nd & Mariposa St),NaN,Alfred J. Hitchcock Productions,Paramount Pictures,Alfred Hitchcock,Alec Coppel,James Stewart,Kim Novak,Barbara Bel Geddes,NaN,<NA>,POINT (-121.53572 36.84592)
872,Family Plot,1976,Nob Hill,Railroad tycoons like Leland Stanford (founder...,Universal Pictures,Universal Pictures,Alfred Hitchcock,Ernest Lehman,Karen Black,Bruce Dern,Barbara Harris,Nob Hill,3,POINT (-122.41742 37.79101)
1036,Vertigo,1958,"Fairmont Hotel (950 Mason St, Nob Hill)",In 1945 the Fairmont hosted the United Nations...,Alfred J. Hitchcock Productions,Paramount Pictures,Alfred Hitchcock,Alec Coppel,James Stewart,Kim Novak,Barbara Bel Geddes,Nob Hill,3,POINT (-122.41009 37.79239)
1188,Marnie,1964,"On Board the SS President Cleveland, docked at...",NaN,Universal Pictures,Universal Pictures,Alfred Hitchcock,Jay Presson Allen,Tippi Hedren,Sean Connery,Martin Gabel,Mission Bay,6,POINT (-122.38489 37.77393)
1362,Family Plot,1976,"Fairmont Hotel (950 Mason St, Nob Hill)",In 1945 the Fairmont hosted the United Nations...,Universal Pictures,Universal Pictures,Alfred Hitchcock,Ernest Lehman,Karen Black,Bruce Dern,Barbara Harris,Nob Hill,3,POINT (-122.41009 37.79239)
